# EXP-9 — Cross Encoder Reranking

Bi-encoders (used in vector search) encode query and document separately, which is fast but approximate. Cross encoders process both together and produce much more accurate relevance scores. We retrieve 10 candidates with the fast bi-encoder, then rerank precisely with the cross encoder.

## Setup

In [6]:
import os
import sys 
import time
import mlflow
import pandas as pd
from dotenv import load_dotenv
from datasets import Dataset

load_dotenv()

# using the new-era langchain packages, not the old monolith
from langchain_community.document_loaders import DirectoryLoader,TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
import warnings
warnings.filterwarnings("ignore")

print("all imports loaded")


all imports loaded


In [2]:
sys.path.append(os.path.abspath(".."))
from data.policies.qa_dataset import dataset

In [3]:
# langsmith traces every chain.invoke() automatically once these env vars are set
# you will see each run in your langsmith project dashboard in real time
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "HR-RAG-Experiments"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")

print("langsmith tracing is active")


langsmith tracing is active


In [4]:
# set MLFLOW_TRACKING_URI in your .env to point to a remote mlflow server
# if running locally first: mlflow server --host 0.0.0.0 --port 5000
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("HR_RAG_Experiments")

print(f"mlflow tracking uri: {MLFLOW_TRACKING_URI}")


mlflow tracking uri: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow


## Load and Prepare Data

In [7]:
# adjust the path and glob pattern to match your folder structure
loader = DirectoryLoader("../data/policies", glob="**/*.md", loader_cls=TextLoader)
docs = loader.load()

print(f"loaded {len(docs)} documents")


loaded 5 documents


In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(docs)
print(f"total chunks: {len(chunks)}")


total chunks: 98


In [10]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
db = FAISS.from_documents(chunks, embeddings)

print("faiss vector store ready")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 648.10it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


faiss vector store ready


In [11]:
print(f"eval set: {len(dataset['question'])} question-answer pairs")

eval set: 10 question-answer pairs


In [12]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

In [13]:
prompt = ChatPromptTemplate.from_template("""
You are an HR assistant. Use only the context below to answer the question.
If the answer is not present in the context, say you do not know.

Context:
{context}

Question: {question}

Answer:""")


## Helper Functions

In [14]:
def format_docs(docs):
    # joins retrieved chunks into a single string that goes into the prompt context
    return "\n\n".join(doc.page_content for doc in docs)


In [15]:
from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig

ragas_llm = LangchainLLMWrapper(llm, run_config=RunConfig(max_workers=1))

In [16]:
import nest_asyncio
nest_asyncio.apply()

from datasets import Dataset, Features, Sequence, Value

def evaluate_rag(chain, get_docs_fn, dataset):
    answers = []
    contexts = []

    for question in dataset["question"]:
        try:
            answer = chain.invoke(question)
            
            # ensure answer is string
            answers.append(str(answer))

            docs = get_docs_fn(question)
            
            # force pure list of strings (IMPORTANT)
            contexts.append([str(d.page_content) for d in docs])

        except Exception as e:
            print(f"  error on: {question[:60]} => {e}")
            answers.append("error")
            contexts.append(["error"])

    #  FORCE correct schema (THIS FIXES YOUR ERROR)
    features = Features({
        "question": Value("string"),
        "answer": Value("string"),
        "contexts": Sequence(Value("string")),  
        "ground_truth": Value("string"),
    })

    eval_dataset = Dataset.from_dict({
        "question": dataset["question"],
        "answer": answers,
        "contexts": contexts,
        "ground_truth": dataset["answer"],
    }, features=features)

    run_cfg = RunConfig(max_workers=1, timeout=120)

    result = evaluate(
        eval_dataset,
        metrics=[faithfulness, context_precision, context_recall],
        llm=ragas_llm,
        embeddings=embeddings,
        run_config=run_cfg,
        raise_exceptions=False,
    )

    return result

In [17]:
def measure_latency(chain, test_q="What is the leave policy?"):
    start = time.time()
    chain.invoke(test_q)
    return round(time.time() - start, 3)

In [18]:
def log_to_mlflow(run_name, result, latency, retriever_type, top_k=3, extra_params=None):
    with mlflow.start_run(run_name=run_name):
        if hasattr(result, "to_pandas"):
           
            scores = result.to_pandas().mean(numeric_only=True).to_dict()
        else:
            scores = dict(result)

        for k, v in scores.items():
            try:
                mlflow.log_metric(k, float(v))
            except Exception:
                pass

        mlflow.log_metric("latency_seconds", latency)
        mlflow.log_param("retriever_type", retriever_type)
        mlflow.log_param("top_k", top_k)

        if extra_params:
            for k, v in extra_params.items():
                mlflow.log_param(k, str(v))

## Experiment-Specific Imports

In [19]:
from sentence_transformers import CrossEncoder


## Experiment

In [20]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
base_retriever = db.as_retriever(search_kwargs={"k": 10})

def cross_encoder_retrieve(question):
    docs = base_retriever.invoke(question)
    # score each (question, doc) pair jointly for accurate relevance
    pairs = [[question, doc.page_content] for doc in docs]
    scores = cross_encoder.predict(pairs)
    # sort descending by score, keep top 3
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in ranked[:3]]

chain = (
    {
        "context": RunnableLambda(cross_encoder_retrieve) | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke("What is the leave policy?"))


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 5215.87it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


This Leave Management Policy outlines the types of leaves available to all full-time, part-time, and contractual employees of TechCorp India Pvt. Ltd.


In [21]:
result = evaluate_rag(chain, cross_encoder_retrieve, dataset)
latency = measure_latency(chain)

Evaluating:  30%|███       | 9/30 [03:52<07:06, 20.32s/it]Runner in Executor raised an exception
Traceback (most recent call last):
  File "c:\projects\Rag-project\.venv\Lib\site-packages\groq\_base_client.py", line 1556, in request
    response.raise_for_status()
  File "c:\projects\Rag-project\.venv\Lib\site-packages\httpx\_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '429 Too Many Requests' for url 'https://api.groq.com/openai/v1/chat/completions'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\user\AppData\Local\Programs\Python\Python312\Lib\asyncio\tasks.py", line 520, in wait_for
    return await fut
           ^^^^^^^^^
  File "c:\projects\Rag-project\.venv\Lib\site-packages\ragas\metrics\_context_recall.py", line 169, i

In [22]:
result,latency

({'faithfulness': 0.7870, 'context_precision': 0.8148, 'context_recall': 0.5267},
 0.919)

In [23]:
log_to_mlflow(
    run_name="EXP-9-CROSS-ENCODER",
    result=result,
    latency=latency,
    retriever_type="cross-encoder-rerank",
    top_k=3,
    extra_params={"initial_k": 10, "cross_encoder_model": "ms-marco-MiniLM-L-6-v2"},
)


🏃 View run EXP-9-CROSS-ENCODER at: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow/#/experiments/1/runs/40c02c4762cb4c7ea02c8b72e742f507
🧪 View experiment at: https://dagshub.com/DataShoaib/hr-policy-chatbot-rag.mlflow/#/experiments/1
